# MedTrack_DV – Milestone 1
## Notebook 04: Data Validation
**Purpose:** Validate all cleaned and normalized datasets before KPI engineering. Check row counts, keys, relationships, and data quality targets.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'

# Load all normalized datasets
df_patients       = pd.read_csv(PROCESSED + 'normalized_patients.csv')
df_bed_records    = pd.read_csv(PROCESSED + 'normalized_bed_records.csv')
df_department     = pd.read_csv(PROCESSED + 'normalized_departments.csv')
df_doctor         = pd.read_csv(PROCESSED + 'normalized_doctors.csv')
df_nurse          = pd.read_csv(PROCESSED + 'normalized_nurses.csv')
df_ward           = pd.read_csv(PROCESSED + 'normalized_wards.csv')
df_beds_patients  = pd.read_csv(PROCESSED + 'normalized_beds_patients.csv')
df_beds_services  = pd.read_csv(PROCESSED + 'normalized_beds_services.csv')
df_readmission    = pd.read_csv(PROCESSED + 'normalized_readmission.csv')
df_healthcare     = pd.read_csv(PROCESSED + 'normalized_healthcare.csv')

print('All normalized datasets loaded for validation!')

## Step 1: Row Count Validation

In [ ]:
datasets = {
    'HMIS Patients':       df_patients,
    'HMIS BedRecords':     df_bed_records,
    'HMIS Departments':    df_department,
    'HMIS Doctors':        df_doctor,
    'HMIS Nurses':         df_nurse,
    'HMIS Wards':          df_ward,
    'Beds Patients':       df_beds_patients,
    'Beds Services':       df_beds_services,
    'Readmission Data':    df_readmission,
    'Healthcare Dataset':  df_healthcare
}

print('=== ROW COUNT VALIDATION ===')
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    print(f'{name}: {len(df):,} rows | Duplicates: {dupes}')

## Step 2: Primary Key Validation

In [ ]:
def validate_primary_key(df, key_col, name):
    if key_col not in df.columns:
        print(f'⚠️  {name}: Column "{key_col}" NOT FOUND')
        return
    nulls    = df[key_col].isnull().sum()
    dupes    = df[key_col].duplicated().sum()
    status   = '✅' if nulls == 0 and dupes == 0 else '❌'
    print(f'{status} {name} [{key_col}]: Nulls={nulls} | Duplicates={dupes}')

print('=== PRIMARY KEY VALIDATION ===')
validate_primary_key(df_patients,    'patient_id',    'HMIS Patients')
validate_primary_key(df_bed_records, 'admission_id',  'HMIS BedRecords')
validate_primary_key(df_department,  'dept_id',       'HMIS Departments')
validate_primary_key(df_doctor,      'doct_id',       'HMIS Doctors')
validate_primary_key(df_nurse,       'nurse_id',      'HMIS Nurses')
validate_primary_key(df_ward,        'ward_no',       'HMIS Wards')
validate_primary_key(df_beds_patients,'patient_id',   'Beds Patients')

## Step 3: Foreign Key Relationship Validation

In [ ]:
def validate_fk(parent_df, parent_key, child_df, child_key, parent_name, child_name):
    if parent_key not in parent_df.columns or child_key not in child_df.columns:
        print(f'⚠️  Cannot validate {parent_name} -> {child_name}: column missing')
        return
    parent_ids = set(parent_df[parent_key].dropna())
    child_ids  = set(child_df[child_key].dropna())
    unmatched  = child_ids - parent_ids
    match_pct  = round((1 - len(unmatched)/max(len(child_ids),1)) * 100, 2)
    status     = '✅' if match_pct >= 95 else '⚠️'
    print(f'{status} {child_name}.{child_key} -> {parent_name}.{parent_key}: {match_pct}% matched ({len(unmatched)} unmatched IDs)')

print('=== FOREIGN KEY VALIDATION ===')
validate_fk(df_patients,   'patient_id', df_bed_records, 'patient_id',  'Patients', 'BedRecords')
validate_fk(df_department, 'dept_id',    df_doctor,      'dept_id',     'Department', 'Doctor')
validate_fk(df_department, 'dept_id',    df_nurse,       'dept_id',     'Department', 'Nurse')
validate_fk(df_ward,       'ward_no',    df_bed_records, 'bed_no',      'Ward', 'BedRecords')

## Step 4: Data Quality Targets Check

In [ ]:
print('=== DATA QUALITY TARGETS ===')
print('Target: Dataset Completeness > 95% | Missing Values < 2%')
print()

for name, df in datasets.items():
    total_cells   = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    missing_pct   = round(missing_cells / total_cells * 100, 2)
    completeness  = round(100 - missing_pct, 2)
    status        = '✅' if missing_pct < 2 else '⚠️'
    print(f'{status} {name}: Completeness={completeness}% | Missing={missing_pct}%')

## Step 5: Outlier Validation

In [ ]:
print('=== OUTLIER VALIDATION ===')

# Length of Stay validation
if 'length_of_stay_days' in df_bed_records.columns:
    los = df_bed_records['length_of_stay_days']
    invalid = los[(los < 0) | (los > 365)]
    print(f'BedRecords LOS: Min={los.min()} | Max={los.max()} | Invalid count={len(invalid)}')

# Healthcare LOS
if 'length_of_stay_days' in df_healthcare.columns:
    los = df_healthcare['length_of_stay_days'].dropna()
    invalid = los[(los < 0) | (los > 365)]
    print(f'Healthcare LOS: Min={los.min()} | Max={los.max()} | Invalid count={len(invalid)}')

# Age validation
if 'age' in df_readmission.columns:
    age = df_readmission['age'].dropna()
    invalid_age = age[(age < 0) | (age > 120)]
    print(f'Readmission Age: Min={age.min()} | Max={age.max()} | Invalid count={len(invalid_age)}')

# Bed utilization
if 'bed_utilization_rate' in df_beds_services.columns:
    util = df_beds_services['bed_utilization_rate'].dropna()
    invalid_util = util[(util < 0) | (util > 100)]
    print(f'Bed Utilization: Min={util.min()}% | Max={util.max()}% | Invalid count={len(invalid_util)}')

## Step 6: Dataset Validation Summary Report

In [ ]:
print('===================================================')
print('   MEDTRACK_DV – DATASET VALIDATION REPORT')
print('===================================================')
print()
print('Dataset 1 – Hospital Management System (HMIS)')
print(f'  Patients:     {len(df_patients):,} records')
print(f'  Admissions:   {len(df_bed_records):,} records')
print(f'  Departments:  {len(df_department):,} records')
print(f'  Doctors:      {len(df_doctor):,} records')
print(f'  Nurses:       {len(df_nurse):,} records')
print(f'  Wards:        {len(df_ward):,} records')
print()
print('Dataset 2 – Hospital Beds Management')
print(f'  Patients:     {len(df_beds_patients):,} records')
print(f'  Services:     {len(df_beds_services):,} records')
print()
print('Dataset 3 – Readmission Data')
print(f'  Admissions:   {len(df_readmission):,} records')
print()
print('Dataset 4 – Healthcare Dataset')
print(f'  Records:      {len(df_healthcare):,} records')
print()
print('Status: ✅ All datasets validated and ready for KPI Engineering')
print('===================================================')

## ✅ Data Validation Complete!
Proceed to **05_kpi_engineering.ipynb**